# Lab | Chains in LangChain

## Outline

* LLMChain
* Sequential Chains
  * SimpleSequentialChain
  * SequentialChain
* Router Chain

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [42]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
HUGGINGFACEHUB_API_TOKEN = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [6]:
%pip install "numpy>=1.26,<2" "pandas>=2.1,<3"

  Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl.metadata (114 kB)
Using cached numpy-1.26.4-cp311-cp311-macosx_11_0_arm64.whl (14.0 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 31.3 MB/s  0:00:00eta 0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
  Attempting uninstall: pandas╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [numpy]
    Found existing installation: pandas 3.1.0━━━━━━━━━━━━━━━━━ 1/3 [numpy]
    Uninstalling pandas-3.1.0:m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [numpy]
      Successfully uninstalled pandas-3.1.0╸━━━━━━━━━━━━━ 2/3 [pandas]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]
Note: you may need to restart the kernel to use updated packages.


In [7]:
import pandas as pd
df = pd.read_csv('./data/Data.csv')

In [15]:
df.head()

,Product,Review
0,Queen Size Sheet Set,I ordered a king size set. My only criticism w...
1,Waterproof Phone Pouch,"I loved the waterproof sac, although the openi..."
2,Luxury Air Mattress,This mattress had a small hole in the top of i...
3,Pillows Insert,This is the best throw pillow fillers on Amazo...
4,Milk Frother Handheld\n,I loved this product. But they only seem to l...


## LLMChain

In [9]:
!pip install langchain_community

In [10]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.chains import LLMChain

In [15]:
#Replace None by your own value and justify
llm = ChatOpenAI(model = 'gpt-6-luna', temperature=1)
# temperature 1 was chosen cuz task involves creative text generation but also should not be too random


In [12]:
prompt = ChatPromptTemplate.from_template('describe a product {product}, including its main features, benefits')


In [16]:
chain = LLMChain(llm=llm, prompt=prompt)

In [18]:
product = 'hockey stick'
chain.run(product)

'## Hockey Stick\n\nA lightweight, durable stick designed to support confident passing, shooting, and puck control on the ice.\n\n**Main features**\n- **Composite shaft:** Lightweight construction makes the stick easier to handle and helps players move it quickly.\n- **Textured grip:** Adds traction for a more secure hold during stickhandling and shooting.\n- **Reinforced blade:** Built to withstand regular contact with the puck and ice.\n- **Curved blade:** Helps with puck control, passing, and shooting; curve options vary by model.\n- **Multiple flex and size options:** Lets players choose a stick suited to their height, strength, and playing style.\n\n**Benefits**\n- Easier handling and quicker movements\n- More control when receiving and passing the puck\n- A comfortable, secure grip\n- Options for different ages, positions, and skill levels\n\nChoose the stick’s length, flex, blade curve, and handedness to match the player.'

## SimpleSequentialChain

In [19]:
from langchain.chains import SimpleSequentialChain

In [28]:
llm = ChatOpenAI(model = 'gpt-6-luna', temperature=1)

# prompt template 1
first_prompt = ChatPromptTemplate.from_template(
    'describe a product {product}, including its main features, benefits'
)
# Chain 1
chain_one = LLMChain(llm=llm, prompt=first_prompt)

In [29]:

# prompt template 2
second_prompt = ChatPromptTemplate.from_template(
    'create a short and catchy advertisement for {product_name}'
)
# chain 2
chain_two = LLMChain(llm=llm, prompt=second_prompt)

In [30]:
overall_simple_chain = SimpleSequentialChain(chains=[chain_one, chain_two],
                                             verbose=True
                                            )

In [31]:
overall_simple_chain.run(product)

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


### Hockey Stick

Designed for control, speed, and reliable performance, this hockey stick helps players handle the puck confidently and release shots quickly. Its lightweight composite construction supports fast movement, while a balanced feel makes it comfortable to use for passing, shooting, and stickhandling.

**Main features**
- **Lightweight composite shaft:** Easier to maneuver and helps reduce arm fatigue.
- **Responsive flex:** Loads during a shot and returns energy to support a quicker release.
- **Reinforced blade:** Built to withstand repeated contact with pucks and the playing surface.
- **Textured grip:** Helps keep hands steady during play.
- **Balanced design:** Supports comfortable handling and accurate puck control.

**Benefits**
- Move the stick quickly in tight spaces.
- Shoot and pass with greater confidence.
- Maintain a secure grip during fast-paced play.
- Enjoy a versatile stick for different on-ice situations.

The ideal flex, curve, length, and handedness dep

'**Own the ice. Control every play.**  \nLightweight speed, a responsive release, and a steady grip—all in one balanced hockey stick. Pass, shoot, and stickhandle with confidence. **Ready for your next shift?**'

**Repeat the above twice for different products**

## SequentialChain

In [32]:
from langchain.chains import SequentialChain

In [33]:
llm = ChatOpenAI(model = 'gpt-6-luna', temperature=1)


first_prompt = ChatPromptTemplate.from_template(
  'translate the following customer review into spanish: {review}'
)

chain_one = LLMChain(llm=llm, prompt=first_prompt, 
                     output_key='translated_review' #Give a name to your output
                    )


In [34]:
second_prompt = ChatPromptTemplate.from_template(
    'summarize the following customer review: {review}'
)

chain_two = LLMChain(llm=llm, prompt=second_prompt, 
                     output_key='summarized_review' #give a name to this output
                    )


In [35]:
# prompt template 3: translate to english or other language
third_prompt = ChatPromptTemplate.from_template(
    'translate the following review into Russian:{review}'
)
# chain 3: input= Review and output= language
chain_three = LLMChain(llm=llm, prompt=third_prompt,
                       output_key='translated_review_2'
                      )


In [36]:

# prompt template 4: follow up message that take as inputs the two previous prompts' variables
fourth_prompt = ChatPromptTemplate.from_template(
        'write a short and polite response to the customer based on the summary:{summarized_review} and this translated review: {translated_review}'
)
chain_four = LLMChain(llm=llm, prompt=fourth_prompt,
                      output_key='follow_up_message'
                     )


In [40]:

overall_chain = SequentialChain(
    chains=[chain_one, chain_two, chain_three, chain_four],
    input_variables=['review'],
    output_variables=[
        'translated_review',
        'summarized_review',
        'follow_up_message',
        'translated_review_2'
    ],
    verbose=True
)

In [41]:
review = df.Review[5]
overall_chain(review)

/var/folders/8_/_4n26j015wl42jtx0x78jr580000gn/T/ipykernel_88839/1992003631.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  overall_chain(review)
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



> Finished chain.


{'review': "Je trouve le goût médiocre. La mousse ne tient pas, c'est bizarre. J'achète les mêmes dans le commerce et le goût est bien meilleur...\nVieux lot ou contrefaçon !?",
 'translated_review': 'El sabor me parece mediocre. La espuma no dura, qué raro. Compro los mismos en las tiendas y saben mucho mejor… ¿Un lote viejo o una falsificación?',
 'summarized_review': 'The customer says the taste is mediocre and the foam disappears quickly, unlike the same product bought in stores. They suspect it may be an old batch or counterfeit.',
 'follow_up_message': 'Lamentamos que el sabor y la espuma no hayan estado a la altura. Nos gustaría revisar la frescura y el origen del producto; ¿podría enviarnos su número de pedido y el código de lote? Con gusto le ayudaremos.',
 'translated_review_2': 'Вкус, на мой взгляд, посредственный. Пена не держится — странно. Покупаю такие же в обычном магазине, и там вкус намного лучше… Старая партия или подделка?!'}

**Repeat the above twice for different products or reviews**

## Router Chain

In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise\
and easy to understand manner. \
When you don't know the answer to a question you admit\
that you don't know.

Here is a question:
{input}"""


math_template = """You are a very good mathematician. \
You are great at answering math questions. \
You are so good because you are able to break down \
hard problems into their component parts, 
answer the component parts, and then put them together\
to answer the broader question.

Here is a question:
{input}"""

history_template = """You are a very good historian. \
You have an excellent knowledge of and understanding of people,\
events and contexts from a range of historical periods. \
You have the ability to think, reflect, debate, discuss and \
evaluate the past. You have a respect for historical evidence\
and the ability to make use of it to support your explanations \
and judgements.

Here is a question:
{input}"""


computerscience_template = """ You are a successful computer scientist.\
You have a passion for creativity, collaboration,\
forward-thinking, confidence, strong problem-solving capabilities,\
understanding of theories and algorithms, and excellent communication \
skills. You are great at answering coding questions. \
You are so good because you know how to solve a problem by \
describing the solution in imperative steps \
that a machine can easily interpret and you know how to \
choose a solution that has a good balance between \
time complexity and space complexity. 

Here is a question:
{input}"""

biology_template = """You are an excellent biologist. \
You have a deep understanding of living organisms, \
from the molecular and cellular level to entire ecosystems. \
You are skilled at observing patterns in nature, analyzing biological data, \
and explaining complex processes like evolution, genetics, physiology, and ecology. \
You can clearly communicate how life functions and adapts, \
and you make connections between different biological concepts \
to answer challenging questions.

Here is a question:
{input}"""

In [ ]:
prompt_infos = [
    {
        "name": "physics", 
        "description": "Good for answering questions about physics", 
        "prompt_template": physics_template
    },
    {
        "name": "math", 
        "description": "Good for answering math questions", 
        "prompt_template": math_template
    },
    {
        "name": "History", 
        "description": "Good for answering history questions", 
        "prompt_template": history_template
    },
    {
        "name": "computer science", 
        "description": "Good for answering computer science questions", 
        "prompt_template": computerscience_template
    },
    {
        "name": "biology",
        "description": "Good for answering biology questions",
        "prompt_template": biology_template
    }
]

In [ ]:
from langchain.chains.router import MultiPromptChain
from langchain.chains.router.llm_router import LLMRouterChain,RouterOutputParser
from langchain.prompts import PromptTemplate

In [ ]:
llm = ChatOpenAI(temperature=0)

In [ ]:
destination_chains = {}
for p_info in prompt_infos:
    name = p_info["name"]
    prompt_template = p_info["prompt_template"]
    prompt = ChatPromptTemplate.from_template(template=prompt_template)
    chain = LLMChain(llm=llm, prompt=prompt)
    destination_chains[name] = chain  
    
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]
destinations_str = "\n".join(destinations)

In [ ]:
default_prompt = ChatPromptTemplate.from_template("{input}")
default_chain = LLMChain(llm=llm, prompt=default_prompt)

In [ ]:
MULTI_PROMPT_ROUTER_TEMPLATE = """Given a raw text input to a \
language model select the model prompt best suited for the input. \
You will be given the names of the available prompts and a \
description of what the prompt is best suited for. \
You may also revise the original input if you think that revising\
it will ultimately lead to a better response from the language model.

<< FORMATTING >>
Return a markdown code snippet with a JSON object formatted to look like:
```json
{{{{
    "destination": string \ name of the prompt to use or "DEFAULT"
    "next_inputs": string \ a potentially modified version of the original input
}}}}
```

REMEMBER: "destination" MUST be one of the candidate prompt \
names specified below OR it can be "DEFAULT" if the input is not\
well suited for any of the candidate prompts.
REMEMBER: "next_inputs" can just be the original input \
if you don't think any modifications are needed.

<< CANDIDATE PROMPTS >>
{destinations}

<< INPUT >>
{{input}}

<< OUTPUT (remember to include the ```json)>>"""

In [ ]:
router_template = MULTI_PROMPT_ROUTER_TEMPLATE.format(
    destinations=destinations_str
)
router_prompt = PromptTemplate(
    template=router_template,
    input_variables=["input"],
    output_parser=RouterOutputParser(),
)

router_chain = LLMRouterChain.from_llm(llm, router_prompt)

In [ ]:
chain = MultiPromptChain(router_chain=router_chain, 
                         destination_chains=destination_chains, 
                         default_chain=default_chain, verbose=True
                        )

In [ ]:
chain.run("What is black body radiation?")

In [ ]:
chain.run("what is 2 + 2")

In [ ]:
chain.run("Why does every cell in our body contain DNA?")

**Repeat the above at least once for different inputs and chains executions - Be creative!**